# Data structures

The two primary data objects in v0.9 are:

| Object | Backing | Holds |
| --- | --- | --- |
| `xftsim.struct.DenseHaplotypeArray` | 3D `np.ndarray` of shape `(n, m, 2)` (maternal/paternal) | Haplotypes for one generation |
| `xftsim.struct.PhenotypeArray` | `dict[str, np.ndarray]` of length-`n` arrays | All phenotypic components for one generation |

Both carry an `xftsim.struct.SampleMeta` describing the rows (`iid`,
`fid`, `sex`, `generation`). `DenseHaplotypeArray` additionally carries
a `VariantMeta` describing the columns (`vid`, `chrom`, `pos_bp`,
`pos_cM`, `af`, `zero_allele`, `one_allele`).

This is a **departure from the legacy interface**, which used
`xarray.DataArray` subclasses with a custom `.xft` accessor. The new
classes implement just the methods we actually need
(`matvec`/`standardized_matvec`, `subset`, allele frequency
properties, etc.) and store data as plain numpy.

There is also a `GraphHaplotypeOperator` that wraps a `pygrgl` GRG and
exposes the same `matvec` / `rmatvec` / `standardized_matvec`
interface — useful when you don't want to materialise an `(n, m)`
matrix. Both `DenseHaplotypeArray` and `GraphHaplotypeOperator`
inherit from `HaplotypeOperator`, which is what `Architecture.compute()`
actually accepts. `GraphHaplotypeOperator` also supports GRG-native
meiosis via a bubble-insertion algorithm, so offspring remain
graph-backed across generations without dense materialization.

First, let's run a small simulation so we have something to inspect.

In [ ]:
import numpy as np
import pandas as pd
import xftsim as xft

xft.config.print_durations_threshold = 10.  # reduce verbosity
np.random.seed(123)

hap = xft.founders.founder_haplotypes_uniform_AFs(n=400, m=100)
eff = xft.effect.AdditiveEffects.from_h2(h2=0.5, m=100, seed=1)
arch = xft.arch.Architecture(
    formula='''
    height.G ~ genetic(eff)
    height.E ~ noise(0.5)
    height   ~ height.G + height.E
    ''',
    effects={'eff': eff},
)
rmap = xft.reproduce.RecombinationMap.from_haplotypes(hap, p=0.1)
mating = xft.mate.LinearAssortativeMating(component_names=['height'], r=0.5)

sim = xft.sim.Simulation(
    founder_haplotypes=hap,
    architecture=arch,
    recombination_map=rmap,
    mating_regime=mating,
    statistics=[xft.stats.SampleStatistics()],
    seed=42,
)
sim.run(n_generations=2)

hap_curr = sim.haplotype_history[sim.generation]
pheno_curr = sim.phenotype_history[sim.generation]


## Phenotype arrays

A `PhenotypeArray` is essentially a `dict[str, np.ndarray]` plus a
`SampleMeta`. Keys are phenotype-component names; arrays are
length-`n`.


In [ ]:
pheno_curr

In [ ]:
list(pheno_curr.keys)

Single-column access by name:

In [ ]:
pheno_curr['height']

Sample metadata (rows):

In [ ]:
pheno_curr.samples

The 'sum' component (`height` here) is just the sum of `height.G` and
`height.E`, as the formula says:


In [ ]:
np.allclose(pheno_curr['height'],
            pheno_curr['height.G'] + pheno_curr['height.E'])


Subsetting takes integer indices (rather than a dict of dimension
labels as in the legacy API):


In [ ]:
n = pheno_curr.samples.n
keep = np.arange(n // 2)
pheno_sub = pheno_curr.subset(keep)
pheno_sub


### Conversion to pandas

There is no `.as_pd()` accessor any more — but converting is one line:


In [ ]:
df = pd.DataFrame({k: pheno_curr[k] for k in pheno_curr.keys})
df.insert(0, 'iid', pheno_curr.samples.iid)
df.insert(1, 'fid', pheno_curr.samples.fid)
df.head()


## Haplotype arrays

A `DenseHaplotypeArray` is a 3D `int8` array of shape `(n, m, 2)`,
where `[:, :, 0]` is the maternal haplotype and `[:, :, 1]` is the
paternal haplotype.


In [ ]:
hap_curr

In [ ]:
hap_curr.genotypes.shape  # (n, m, 2)

### Sample and variant metadata


In [ ]:
hap_curr.samples

In [ ]:
hap_curr.variants

### Empirical and ancestral allele frequencies

Empirical allele frequencies are available on the array; ancestral
frequencies live on `variants.af`:


In [ ]:
pd.DataFrame.from_dict(dict(
    ancestral=hap_curr.variants.af,
    empirical=hap_curr.af_empirical,
)).T


### Matrix-vector operations

`DenseHaplotypeArray` (and any `HaplotypeOperator`) supports both
plain `matvec`/`rmatvec` and standardized versions that act on
`(G - 2p) / sqrt(2 p (1 - p))` without materialising the standardized
matrix. The architecture uses these internally to compute genetic
components:


In [ ]:
m = hap_curr.m
v = np.random.randn(m)

g_raw = hap_curr.matvec(v)
g_std = hap_curr.standardized_matvec(v)
g_raw.shape, g_std.shape


There is also a `StandardizedHaplotypeOperator` wrapper that lazily
exposes standardized operations on an underlying operator, used by
`HasemanElstonEstimator` to avoid building the GRM explicitly.

### Genetic maps

The legacy `xftsim.struct.GeneticMap.from_pyrho_maps()` constructor is
not (yet) supported on the new `GeneticMap`. We skip realistic
recombination in this user guide; see the recombination-maps tutorial
for what is currently exposed.
